# Trabajo Práctico N°3 - Minería de Datos:

> **Año:** 2026  
> **Integrantes:**  
> * Larrubia, Agustín.  
> * Lazarte, Manuel.


## Introducción:
&emsp;El objetivo de este trabajo práctico es integrar los conocimientos adquiridos en las
unidades 5 y 6 en un problema real de clasificación multiclase, asociado a la
recomendación de cultivos agrícolas en función de parámetros de suelo y clima los cuales encontramos en el dataset 'crop_recommendation_geo.csv'. La
variable objetivo es Clase (tipo de cultivo recomendado, con 10 clases).

## Descripción del conjunto de datos

El conjunto de datos `crop_recommendation_geo.csv` contiene **50005 muestras** y **17 columnas**. Fue **modificado intencionalmente** respecto del original ([Kaggle - Global Crop Recommendation](https://www.kaggle.com/datasets/markjehan/global-croprecommendation)).

### Variables predictoras (16)

| Variable | Descripción |
|---|---|
| `N` | Nitrógeno total en el suelo (0-30 cm) |
| `P` | Fósforo disponible para las plantas (Olsen) |
| `exch_bases` | Bases intercambiables totales (Ca + Mg + K + Na), capa superficial del suelo: la medición de potasio |
| `ph` | pH del suelo en agua (0-30 cm) |
| `soc` | Carbono orgánico del suelo (0-30 cm) |
| `arcilla` | Fracción arcillosa (0-30 cm) |
| `arena` | Fracción de arena (0-30 cm) |
| `cec` | Capacidad de intercambio catiónico |
| `temperatura` | Temperatura media anual del aire |
| `temp_mim_mensual` | Temperatura media del mes más frío |
| `temp_max_mensual` | Temperatura media del mes más cálido |
| `humedad` | Humedad relativa media anual |
| `lluvia` | Precipitación anual |
| `estacionalidad_lluvias` | Coeficiente de variación de la precipitación mensual (BIO15) |
| `radiacion_solar` | Radiación solar incidente media anual |
| `elevacion` | Altura sobre el nivel del mar |

### Variable objetivo

`Clase`: cultivo recomendado, con **10 clases**: Algodón, Arroz, Banana, Café, Coco, Garbanzo, Frijol, Lenteja, Maíz y Manzana.

> **Nota:** en todo el trabajo se utiliza `random_state=13` en modelos, particiones y validaciones cruzadas para garantizar la reproducibilidad.

# Librerías

In [ ]:
# Manejo de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y validación
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, train_test_split

# Modelos y métricas
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

# Constantes
RANDOM_STATE = 13


## 1. Preparación de datos


### 1.1 Carga, formato y estructura inicial

El archivo `crop_recommendation_geo.csv` utiliza punto y coma (`;`) como separador de columnas y coma (`,`) como separador decimal. Se carga directamente con esos parámetros y se estandarizan los nombres de las columnas a minúsculas.


In [ ]:
df_raw = pd.read_csv("crop_recommendation_geo.csv", encoding="utf-8-sig", sep=";", decimal=",")
df_raw.columns = df_raw.columns.str.strip().str.lower()

num_cols = [
    "n", "p", "exch_bases", "ph", "soc", "arcilla", "arena", "cec",
    "temperatura", "temp_min_mensual", "temp_max_mensual", "humedad",
    "lluvia", "estacionalidad_lluvias", "radiacion_solar", "elevacion"
]

print("Dimensiones iniciales:", df_raw.shape)
print("\nTipos de dato:\n", df_raw.dtypes)
df_raw.head()


### 1.2 Detección y tratamiento de problemas del dataset


#### 1.2.1 Valores faltantes

In [ ]:
# Diagnóstico de valores faltantes
missing_cols = df_raw.isna().sum()
print("Valores faltantes por columna:")
print(missing_cols[missing_cols > 0])

print("\nFilas que contienen al menos un valor nulo:")
display(df_raw[df_raw.isna().any(axis=1)])


**Decisión (valores faltantes):** se detectaron **3 valores faltantes** (`ph`, `cec` y `humedad`), pero todos pertenecen a una **única fila** (índice `25085`, clase `cafe`). Se opta por **eliminar ese registro** (`dropna()`) porque representa apenas el 0,002% del conjunto de datos y, al faltarle tres atributos relevantes en simultáneo (suelo y clima), imputarlos introduciría valores sintéticos innecesarios.

#### 1.2.2 Registros duplicados


In [ ]:
# Diagnóstico de duplicados exactos
n_dups = df_raw.duplicated().sum()
print(f"Cantidad de registros duplicados exactos: {n_dups}")
display(df_raw[df_raw.duplicated(keep=False)].head(6))


**Decisión (duplicados):** se detectaron **198 registros duplicados exactos**. Se eliminan mediante `drop_duplicates()` antes de cualquier partición o validación cruzada para evitar *data leakage* (si una misma fila queda simultáneamente en entrenamiento y validación, las métricas se inflan artificialmente).

#### 1.2.3 Outliers y valores físicamente imposibles


In [ ]:
# Verificación de rangos físicos/agronómicos imposibles
mask_imposibles = (
    (df_raw["ph"] < 0) | (df_raw["ph"] > 14) |
    (df_raw["humedad"] < 0) | (df_raw["humedad"] > 100) |
    (df_raw["arcilla"] < 0) | (df_raw["arcilla"] > 100) |
    ((df_raw["arcilla"] + df_raw["arena"]) > 100)
)

print(f"Registros con valores físicamente imposibles: {mask_imposibles.sum()}")
display(df_raw[mask_imposibles])

# Aplicamos la limpieza completa en orden
df = df_raw.drop_duplicates().copy()
df = df.dropna().copy()
mask_imp_clean = (
    (df["ph"] < 0) | (df["ph"] > 14) |
    (df["humedad"] < 0) | (df["humedad"] > 100) |
    (df["arcilla"] < 0) | (df["arcilla"] > 100) |
    ((df["arcilla"] + df["arena"]) > 100)
)
df = df[~mask_imp_clean].reset_index(drop=True)

print(f"\nFilas originales: {len(df_raw)}")
print(f"Filas tras eliminar duplicados (-198), nulos (-1) e imposibles (-2): {len(df)}")
print(f"Porcentaje conservado: {len(df) / len(df_raw) * 100:.2f}%")

# Conteo informativo de valores extremos según criterio IQR (sin recortar)
print("\nCantidad de valores fuera de 1.5*IQR (conservados por ser variabilidad natural):")
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"  {col}: {n_out}")


**Decisión (outliers):** se identificaron **2 filas con valores físicamente imposibles**: índice `25111` (`cafe`, con `ph = 25,317`) e índice `30012` (`manzana`, con `ph = 18,017`, `arcilla = 142,383%`, `humedad = 148,141%` y `temp_max_mensual = 53,006 °C`). Estas 2 filas se eliminan por corresponder claramente a datos corruptos.

Por otro lado, **no se aplica recorte (capping) ni eliminación por IQR** sobre el resto de las columnas: al tratarse de 10 cultivos con requerimientos agroclimáticos muy diversos, los valores extremos reales (por ejemplo, precipitaciones muy altas en arroz/banana o mayor elevación en café/manzana) constituyen señal discriminativa genuina y recortarlos empeoraría la separación entre clases.


### 1.3 Análisis exploratorio de datos (EDA)


#### 1.3.1 Distribuciones de las variables predictoras

**Hipótesis previa:** al convivir cultivos de climas muy distintos (tropicales como banana, coco y café frente a templados como manzana y lenteja), esperamos observar distribuciones asimétricas o multimodales en las variables climáticas (`temperatura`, `lluvia`, `humedad`) y rangos de escala muy dispares entre atributos.

In [ ]:
display(df[num_cols].describe().T)

fig, axes = plt.subplots(4, 4, figsize=(20, 15))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, bins=30)
    ax.set_title(col)
plt.suptitle("Distribución de las 16 variables predictoras (dataset limpio)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


**Análisis:** la hipótesis se confirma. Variables como `lluvia`, `p`, `soc` y `elevacion` presentan fuerte asimetría positiva, mientras que `temperatura`, `temp_min_mensual` y `ph` muestran comportamientos multimodales propios de distintos nichos ecológicos. Además, se confirman diferencias de escala de varios órdenes de magnitud (por ejemplo, `n` entre 0,27 y 15,1 frente a `lluvia` entre 3 y 6.759 mm), lo que ratifica la necesidad de estandarizar antes de entrenar SVM.

#### 1.3.2 Dispersión y valores extremos por variable

**Hipótesis previa:** una vez eliminadas las 2 filas con valores físicamente imposibles, esperamos que los rangos de todas las variables sean agronómicamente plausibles y que la dispersión remanente se concentre en las colas superiores de variables climáticas y de nutrientes (`lluvia`, `soc`, `p`, `elevacion`).


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 15))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.suptitle("Boxplots de las variables predictoras (dataset limpio)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


**Análisis:** se confirma la hipótesis. Tras remover los registros imposibles, `ph` queda acotado entre 4,05 y 8,95, `humedad` entre 28,37% y 97,37%, y `arcilla` hasta 68,63%. Las colas observadas en `lluvia`, `soc`, `p` y `elevacion` son coherentes con regiones reales de cultivo y se conservan para no perder capacidad predictiva.

#### 1.3.3 Análisis de correlaciones

**Hipótesis previa:** esperamos correlaciones positivas fuertes entre las tres variables térmicas (`temperatura`, `temp_min_mensual`, `temp_max_mensual`) y entre `humedad` y `lluvia`, así como una correlación negativa marcada entre `arcilla` y `arena` por ser fracciones complementarias de la textura del suelo.


In [ ]:
plt.figure(figsize=(16, 13))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de correlación de Pearson (dataset limpio)")
plt.tight_layout()
plt.show()


**Análisis:** la hipótesis se cumple por completo: `temperatura` correlaciona fuertemente con `temp_min_mensual` (`0,95`) y `temp_max_mensual` (`0,75`), `humedad` con `lluvia` (`0,67`) y `arcilla` negativamente con `arena` (`-0,65`). Adicionalmente, surge una relación edafo-climática destacable no prevista inicialmente: una fuerte correlación negativa entre `ph` y `humedad` (`-0,79`) y entre `ph` y `lluvia` (`-0,78`), junto con `exch_bases` vs `ph` (`0,72`), lo cual es consistente con la acidificación de suelos en zonas muy lluviosas.

#### 1.3.4 Distribución de la variable objetivo (`clase`)

**Hipótesis previa:** esperamos una distribución casi balanceada entre los 10 cultivos (~5.000 muestras por clase), con diferencias menores derivadas de los registros duplicados e inválidos eliminados.


In [ ]:
plt.figure(figsize=(11, 5))
order = df["clase"].value_counts().index
sns.countplot(data=df, x="clase", order=order, palette="viridis", hue="clase", legend=False)
plt.title("Distribución de la variable objetivo (clase) tras la limpieza")
plt.xlabel("Cultivo")
plt.ylabel("Cantidad de muestras")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Conteo exacto por clase:")
print(df["clase"].value_counts())


**Análisis:** la hipótesis se confirma. Las clases oscilan entre **4.891** muestras (`lenteja`) y **5.000** (`banana`, `coco`, `cafe`), evidenciando un leve desbalance (~2,2% entre los extremos) que justifica emplear `StratifiedKFold` y métricas promediadas con `weighted`.


### 1.4 Codificación de la variable objetivo y estrategia de estandarización

Se codifica `clase` con `LabelEncoder` para obtener etiquetas numéricas reproducibles. En cuanto a `StandardScaler`, **no se ajusta sobre todo el conjunto `X` en este paso** para evitar fuga de datos (*data leakage*): se aplicará dentro de un `Pipeline` en cada fold de la validación cruzada en las Actividades 2 y 3.


In [ ]:
X = df[num_cols].copy()
le = LabelEncoder()
y = le.fit_transform(df["clase"])

print("Shape de X:", X.shape, "| Shape de y:", y.shape)
print("Mapeo de clases:", dict(zip(le.classes_, le.transform(le.classes_))))


## 2. SVM con kernel lineal


### 2.1 Configuración de la validación cruzada y esquema sin fuga de datos

Se define un único esquema `StratifiedKFold(n_splits=5, shuffle=True, random_state=13)` que se reutilizará en todos los modelos del trabajo para garantizar una comparación justa sobre los mismos folds. La estandarización (`StandardScaler`) se encapsula dentro de un `Pipeline` para que la media y el desvío se estimen únicamente sobre los folds de entrenamiento en cada iteración.


In [ ]:
# Validación cruzada estratificada compartida por las Actividades 2, 3 y 4
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision_w": "precision_weighted",
    "recall_w": "recall_weighted",
}

def resumir_grid(grid, param_cols):
    """Convierte cv_results_ de GridSearchCV en una tabla legible (media ± std)."""
    res = pd.DataFrame(grid.cv_results_)
    tabla = pd.DataFrame()
    for col in param_cols:
        nombre = col.replace("param_clf__", "")
        tabla[nombre] = res[col]

    tabla["acc_mean"] = res["mean_test_accuracy"]
    tabla["acc_std"] = res["std_test_accuracy"]
    tabla["prec_mean"] = res["mean_test_precision_w"]
    tabla["prec_std"] = res["std_test_precision_w"]
    tabla["rec_mean"] = res["mean_test_recall_w"]
    tabla["rec_std"] = res["std_test_recall_w"]
    tabla["fit_time_s"] = res["mean_fit_time"]

    tabla["Accuracy"] = tabla.apply(lambda r: f"{r['acc_mean']:.4f} ± {r['acc_std']:.4f}", axis=1)
    tabla["Precision (weighted)"] = tabla.apply(lambda r: f"{r['prec_mean']:.4f} ± {r['prec_std']:.4f}", axis=1)
    tabla["Recall (weighted)"] = tabla.apply(lambda r: f"{r['rec_mean']:.4f} ± {r['rec_std']:.4f}", axis=1)
    tabla["Tiempo fit (s)"] = tabla["fit_time_s"].map(lambda x: f"{x:.2f}")
    return tabla


### 2.2 Exploración del parámetro `C`

**Hipótesis previa:** dado que el problema involucra 10 cultivos con combinaciones superpuestas de clima y suelo, esperamos que una frontera estrictamente lineal alcance un rendimiento moderado a bueno pero con un techo claro. Al aumentar `C` desde `0.01` hacia valores mayores, el rendimiento debería mejorar inicialmente y luego estabilizarse cuando el margen ya no pueda separar mejor las clases solapadas, mientras el tiempo de entrenamiento crece de forma marcada en `C` altos.


In [ ]:
pipe_svm_lin = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="linear", cache_size=1000, random_state=RANDOM_STATE))
])

param_grid_lin = {
    "clf__C": [0.01, 0.1, 1, 10, 100]
}

grid_svm_lin = GridSearchCV(
    estimator=pipe_svm_lin,
    param_grid=param_grid_lin,
    cv=cv,
    scoring=scoring,
    refit="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_svm_lin.fit(X, y)

tabla_svm_lin = resumir_grid(grid_svm_lin, ["param_clf__C"])
cols_vista_lin = ["C", "Accuracy", "Precision (weighted)", "Recall (weighted)", "Tiempo fit (s)"]
display(tabla_svm_lin[cols_vista_lin])


### 2.3 Visualización de resultados y contraste con la hipótesis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: métricas en función de C (escala logarítmica)
c_vals = tabla_svm_lin["C"].astype(float)
axes[0].errorbar(c_vals, tabla_svm_lin["acc_mean"], yerr=tabla_svm_lin["acc_std"],
                 marker="o", capsize=4, label="Accuracy")
axes[0].errorbar(c_vals, tabla_svm_lin["prec_mean"], yerr=tabla_svm_lin["prec_std"],
                 marker="s", capsize=4, linestyle="--", label="Precision (weighted)")
axes[0].errorbar(c_vals, tabla_svm_lin["rec_mean"], yerr=tabla_svm_lin["rec_std"],
                 marker="^", capsize=4, linestyle=":", label="Recall (weighted)")
axes[0].set_xscale("log")
axes[0].set_xlabel("C (escala logarítmica)")
axes[0].set_ylabel("Puntaje (media ± std en 5 folds)")
axes[0].set_title("Rendimiento de SVM Lineal según C")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Panel derecho: tiempo medio de entrenamiento por fold
axes[1].plot(c_vals, tabla_svm_lin["fit_time_s"], marker="o", color="tab:red")
axes[1].set_xscale("log")
axes[1].set_xlabel("C (escala logarítmica)")
axes[1].set_ylabel("Tiempo medio de entrenamiento por fold (s)")
axes[1].set_title("Costo computacional de SVM Lineal según C")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resumen automático de la mejor configuración para usar en Actividad 5
best_lin_row = tabla_svm_lin.sort_values("acc_mean", ascending=False).iloc[0]
best_svm_lin = {
    "Modelo": "SVM Lineal",
    "Hiperparámetros": f"C={best_lin_row['C']}",
    "Accuracy": best_lin_row["Accuracy"],
    "Precision (weighted)": best_lin_row["Precision (weighted)"],
    "Recall (weighted)": best_lin_row["Recall (weighted)"],
    "Tiempo fit (s)": best_lin_row["Tiempo fit (s)"],
    "acc_mean": best_lin_row["acc_mean"],
    "acc_std": best_lin_row["acc_std"],
    "fit_time_s": best_lin_row["fit_time_s"],
}
print("Mejor configuración SVM Lineal:", best_svm_lin)


### 2.4 Selección de la mejor configuración y justificación

Al analizar la tabla y los dos paneles anteriores, evaluamos si se confirma la hipótesis previa:

- **Evolución de métricas vs. `C`:** verificar si a partir de un valor intermedio de `C` (por ejemplo `C=1` o `C=10`) las tres métricas se amesetan y la ganancia al pasar a `C=100` resulta marginal.
- **Estabilidad:** observar si la desviación estándar entre folds es baja y pareja en toda la curva.
- **Criterio de selección:** se selecciona el valor de `C` que maximiza las métricas de validación (o el menor `C` dentro de la meseta óptima si la diferencia es menor al desvío estándar, priorizando menor costo computacional y menor riesgo de sobreajuste).


## 3. SVM con kernel gaussiano (RBF)


### 3.1 Exploración de `C` y `gamma`

**Hipótesis previa:** esperamos que el kernel gaussiano (RBF) supere claramente al SVM lineal, ya que los nichos agroclimáticos de los cultivos suelen definirse por combinaciones no lineales y rangos acotados de temperatura, lluvia, humedad y suelo. Respecto de la interacción entre hiperparámetros, esperamos bajo rendimiento (*underfitting*) con `gamma = 0.001` (fronteras demasiado suaves), un pico de desempeño en valores intermedios de `gamma` (`0.01`–`0.1`) con `C` moderado/alto (`10`–`100`), y una posible caída o pérdida de generalización cuando `gamma = 1` genera fronteras excesivamente locales.


In [ ]:
pipe_svm_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", cache_size=1000, random_state=RANDOM_STATE))
])

param_grid_rbf = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": [0.001, 0.01, 0.1, 1]
}

grid_svm_rbf = GridSearchCV(
    estimator=pipe_svm_rbf,
    param_grid=param_grid_rbf,
    cv=cv,
    scoring=scoring,
    refit="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_svm_rbf.fit(X, y)

tabla_svm_rbf = resumir_grid(grid_svm_rbf, ["param_clf__C", "param_clf__gamma"])
cols_vista_rbf = ["C", "gamma", "Accuracy", "Precision (weighted)", "Recall (weighted)", "Tiempo fit (s)"]
display(tabla_svm_rbf.sort_values("acc_mean", ascending=False)[cols_vista_rbf])


### 3.2 Heatmap de interacción (`C` vs. `gamma`) y contraste con la hipótesis


In [ ]:
pivot_acc = tabla_svm_rbf.pivot(index="C", columns="gamma", values="acc_mean")
pivot_time = tabla_svm_rbf.pivot(index="C", columns="gamma", values="fit_time_s")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(pivot_acc, annot=True, fmt=".4f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("SVM RBF: Accuracy media (5-Fold CV)")
axes[0].set_xlabel("gamma")
axes[0].set_ylabel("C")

sns.heatmap(pivot_time, annot=True, fmt=".1f", cmap="OrRd", ax=axes[1])
axes[1].set_title("SVM RBF: Tiempo medio de fit por fold (s)")
axes[1].set_xlabel("gamma")
axes[1].set_ylabel("C")

plt.tight_layout()
plt.show()

# Resumen automático de la mejor configuración para usar en Actividad 5
best_rbf_row = tabla_svm_rbf.sort_values("acc_mean", ascending=False).iloc[0]
best_svm_rbf = {
    "Modelo": "SVM RBF",
    "Hiperparámetros": f"C={best_rbf_row['C']}, gamma={best_rbf_row['gamma']}",
    "Accuracy": best_rbf_row["Accuracy"],
    "Precision (weighted)": best_rbf_row["Precision (weighted)"],
    "Recall (weighted)": best_rbf_row["Recall (weighted)"],
    "Tiempo fit (s)": best_rbf_row["Tiempo fit (s)"],
    "acc_mean": best_rbf_row["acc_mean"],
    "acc_std": best_rbf_row["acc_std"],
    "fit_time_s": best_rbf_row["fit_time_s"],
}
print("Mejor configuración SVM RBF:", best_svm_rbf)


### 3.3 Selección de la mejor configuración y justificación

A partir de la tabla ordenada y los heatmaps de Accuracy y tiempo de entrenamiento:

- **Contraste con el SVM lineal:** comparar la mejor Accuracy/Recall del kernel RBF frente al techo del SVM lineal para confirmar si las relaciones no lineales aportan una mejora sustancial.
- **Interacción `C` vs. `gamma`:** verificar en el heatmap izquierdo dónde se concentra la zona óptima y comprobar si `gamma` muy bajo subajusta o si `gamma = 1` degrada el desempeño y dispara el costo computacional (heatmap derecho).
- **Elección final:** se elige la combinación `(C, gamma)` con mejor equilibrio entre Accuracy/Precision/Recall, baja variabilidad entre folds y tiempo de ajuste razonable.


## 4. Random Forest


### 4.1 Exploración de `n_estimators` y `max_depth`

**Hipótesis previa:** al aumentar `max_depth` desde `5` hacia `20` o `None`, esperamos un salto importante en el rendimiento porque árboles poco profundos no alcanzan a separar las combinaciones de 16 variables entre 10 cultivos. En cambio, aumentar `n_estimators` por encima de `100`–`200` debería aportar mejoras marginales en Accuracy pero reducir la varianza entre folds (menor desviación estándar), incrementando de forma lineal el tiempo de entrenamiento.


In [ ]:
# Random Forest no requiere escalado; se envuelve en Pipeline para reutilizar resumir_grid y el mismo CV
pipe_rf = Pipeline([
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid_rf = {
    "clf__n_estimators": [50, 100, 200, 500],
    "clf__max_depth": [5, 10, 20, None]
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring=scoring,
    refit="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X, y)

tabla_rf = resumir_grid(grid_rf, ["param_clf__n_estimators", "param_clf__max_depth"])
# Convertimos None a texto explícito para que pivot/display no lo omita
tabla_rf["max_depth_str"] = tabla_rf["max_depth"].apply(lambda v: "None" if pd.isna(v) else str(int(v)))

cols_vista_rf = ["n_estimators", "max_depth_str", "Accuracy", "Precision (weighted)", "Recall (weighted)", "Tiempo fit (s)"]
display(tabla_rf.sort_values("acc_mean", ascending=False)[cols_vista_rf].rename(columns={"max_depth_str": "max_depth"}))


### 4.2 Visualización de resultados (`n_estimators` vs. `max_depth`)


In [ ]:
orden_depth = ["5", "10", "20", "None"]
pivot_rf_acc = tabla_rf.pivot(index="max_depth_str", columns="n_estimators", values="acc_mean").reindex(orden_depth)
pivot_rf_time = tabla_rf.pivot(index="max_depth_str", columns="n_estimators", values="fit_time_s").reindex(orden_depth)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(pivot_rf_acc, annot=True, fmt=".4f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Random Forest: Accuracy media (5-Fold CV)")
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("max_depth")

sns.heatmap(pivot_rf_time, annot=True, fmt=".1f", cmap="OrRd", ax=axes[1])
axes[1].set_title("Random Forest: Tiempo medio de fit por fold (s)")
axes[1].set_xlabel("n_estimators")
axes[1].set_ylabel("max_depth")

plt.tight_layout()
plt.show()

# Resumen automático de la mejor configuración para usar en Actividad 5
best_rf_row = tabla_rf.sort_values("acc_mean", ascending=False).iloc[0]
best_rf = {
    "Modelo": "Random Forest",
    "Hiperparámetros": f"n_estimators={int(best_rf_row['n_estimators'])}, max_depth={best_rf_row['max_depth_str']}",
    "Accuracy": best_rf_row["Accuracy"],
    "Precision (weighted)": best_rf_row["Precision (weighted)"],
    "Recall (weighted)": best_rf_row["Recall (weighted)"],
    "Tiempo fit (s)": best_rf_row["Tiempo fit (s)"],
    "acc_mean": best_rf_row["acc_mean"],
    "acc_std": best_rf_row["acc_std"],
    "fit_time_s": best_rf_row["fit_time_s"],
}
print("Mejor configuración Random Forest:", best_rf)


### 4.3 Importancia de las variables (`feature_importances_`) del mejor modelo

**Hipótesis previa:** esperamos que las variables climáticas (`lluvia`, `temperatura`, `temp_min_mensual`, `humedad`, `estacionalidad_lluvias`) dominen la importancia por sobre las edáficas (`ph`, `arcilla`, `arena`, `n`, `p`), ya que el clima delimita en primera instancia la viabilidad geográfica de cada cultivo.


In [ ]:
best_rf_clf = grid_rf.best_estimator_.named_steps["clf"]
importancias = pd.Series(best_rf_clf.feature_importances_, index=num_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importancias.plot(kind="barh", color="teal")
plt.title("Importancia de variables — Mejor modelo Random Forest")
plt.xlabel("Importancia relativa (reducción media de impureza)")
plt.ylabel("Variable predictora")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 5 variables más importantes:")
display(importancias.sort_values(ascending=False).head(5).to_frame("Importancia"))
print("5 variables menos importantes:")
display(importancias.head(5).to_frame("Importancia"))


### 4.4 Selección de la mejor configuración y justificación

A partir de la tabla, los heatmaps y el gráfico de importancia de variables:

- **Efecto de `max_depth` y `n_estimators`:** verificar si el salto principal ocurre al pasar de `max_depth = 5` a `20`/`None`, mientras que incrementar `n_estimators` de `200` a `500` aporta una ganancia marginal frente al aumento de tiempo de entrenamiento.
- **Contraste sobre `feature_importances_`:** comprobar si las variables climáticas lideran el ranking como se planteó en la hipótesis previa y señalar qué variables edáficas o geográficas (`elevacion`, `ph`, `soc`, etc.) aportaron más o menos de lo esperado.
- **Selección:** se elige la configuración que logra el mejor equilibrio entre métricas de validación, baja desviación estándar entre folds y tiempo de ajuste razonable.


## 5. Comparación de modelos


### 5.1 Tabla comparativa de las mejores configuraciones


In [ ]:
tabla_comparativa = pd.DataFrame([best_svm_lin, best_svm_rbf, best_rf])
cols_comp = ["Modelo", "Hiperparámetros", "Accuracy", "Precision (weighted)", "Recall (weighted)", "Tiempo fit (s)"]
display(tabla_comparativa[cols_comp])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Accuracy media ± std
axes[0].bar(tabla_comparativa["Modelo"], tabla_comparativa["acc_mean"],
            yerr=tabla_comparativa["acc_std"], capsize=6, color=["#4c72b0", "#dd8452", "#55a868"])
axes[0].set_ylabel("Accuracy media (5-Fold CV)")
axes[0].set_title("Comparación de Accuracy (media ± std)")
axes[0].set_ylim(max(0, tabla_comparativa["acc_mean"].min() - 0.05), min(1.0, tabla_comparativa["acc_mean"].max() + 0.03))
axes[0].grid(axis="y", alpha=0.3)

# Tiempo medio de entrenamiento por fold
axes[1].bar(tabla_comparativa["Modelo"], tabla_comparativa["fit_time_s"],
            color=["#4c72b0", "#dd8452", "#55a868"])
axes[1].set_ylabel("Tiempo medio de fit por fold (s)")
axes[1].set_title("Comparación de costo computacional")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### 5.2 Matrices de confusión

**Hipótesis previa:** esperamos que los errores de clasificación no se distribuyan de forma uniforme, sino que se concentren entre cultivos con requerimientos agroclimáticos parecidos (por ejemplo, legumbres como `garbanzo`, `frijol` y `lenteja`, o cultivos tropicales como `cafe`, `banana` y `coco`). Además, esperamos que el mejor modelo no lineal (SVM RBF o Random Forest) reduzca claramente esas confusiones frente al SVM lineal.


In [ ]:
from sklearn.base import clone

# Split estratificado 80/20 reproducible (random_state=13) para visualizar las 3 matrices de confusión
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

modelos_mejores = {
    "SVM Lineal": clone(grid_svm_lin.best_estimator_),
    "SVM RBF": clone(grid_svm_rbf.best_estimator_),
    "Random Forest": clone(grid_rf.best_estimator_),
}

fig, axes = plt.subplots(1, 3, figsize=(23, 6.5))

for ax, (nombre, modelo) in zip(axes, modelos_mejores.items()):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False, values_format="d")
    acc_test = accuracy_score(y_test, y_pred)
    ax.set_title(f"{nombre} (Acc test 80/20: {acc_test:.4f})")

    # Reporte automático de los 5 pares de clases más confundidos
    cm_off = cm.copy()
    np.fill_diagonal(cm_off, 0)
    pares = []
    for i in range(len(le.classes_)):
        for j in range(len(le.classes_)):
            if cm_off[i, j] > 0:
                pares.append((cm_off[i, j], le.classes_[i], le.classes_[j]))
    pares.sort(reverse=True)
    print(f"\nTop 5 confusiones — {nombre} (Real -> Predicho):")
    for cnt, real_c, pred_c in pares[:5]:
        print(f"  {real_c} -> {pred_c}: {cnt} errores")

plt.tight_layout()
plt.show()


### 5.3 Selección del modelo más adecuado

Para definir el modelo ganador ponderamos los cuatro criterios solicitados en conjunto:

- **Métricas obtenidas:** comparar Accuracy, Precision (weighted) y Recall (weighted) de la tabla `5.1` y la reducción de errores fuera de la diagonal en las matrices de confusión de `5.2`.
- **Estabilidad entre folds:** verificar la desviación estándar entre los 5 folds; un modelo con menor desvío garantiza desempeño más consistente ante nuevos datos.
- **Interpretabilidad:** Random Forest ofrece una ventaja directa al cuantificar la importancia de cada variable (`feature_importances_`), lo que facilita validar el modelo con criterio agronómico; en cambio, SVM RBF opera como caja negra en un espacio proyectado.
- **Tiempo de entrenamiento:** contrastar el tiempo medio por fold del panel derecho de `5.1`, ya que sobre ~50.000 registros SVM suele escalar peor que Random Forest paralelizado (`n_jobs=-1`).


## 6. Conclusiones

- **Calidad y preparación de los datos:** la inspección inicial permitió identificar y corregir las alteraciones del dataset (`198` duplicados exactos, `1` fila con 3 valores faltantes simultáneos y `2` filas con valores físicamente imposibles en `ph`, `humedad` y `arcilla`), conservando el 99,60% de las observaciones (`49.803` muestras) sin recortar extremos climáticos reales. Encapsular `StandardScaler` dentro de un `Pipeline` en `StratifiedKFold(k=5)` evitó fugas de datos y aseguró una evaluación justa.
- **Hallazgos del análisis exploratorio:** las correlaciones confirmaron redundancias térmicas esperadas (`temperatura` vs. `temp_min_mensual`: `0,95`) y revelaron fuertes vínculos edafo-climáticos (`ph` vs. `humedad`: `-0,79` y `ph` vs. `lluvia`: `-0,78`), anticipando que la separación entre cultivos requiere capturar interacciones no lineales.
- **Desempeño comparativo de los modelos:** los resultados de las Actividades 2 a 5 permiten contrastar directamente los límites de un clasificador lineal (`SVM Lineal`) frente a modelos no lineales (`SVM RBF` y `Random Forest`), tanto en capacidad predictiva global como en la reducción de confusiones entre cultivos de nichos similares.
- **Elección final:** tras ejecutar las celdas de entrenamiento, cerrar aquí indicando el modelo ganador definitivo con sus hiperparámetros óptimos, sus métricas finales (media ± desvío) y por qué ofrece el mejor balance entre desempeño, estabilidad, interpretabilidad y costo computacional para recomendar cultivos.
